In [1]:
from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

In [2]:
def create_data_model():
    """Stores the data for the problem."""
    data = {}
    data["distance_matrix"] =  [
        # 0    1    2    3    4    5    6    7    8    9   10
        [  0, 548, 776, 696, 582, 274, 502, 194, 308, 194, 536],  # 0 depot
        [548,   0, 684, 308, 194, 502, 730, 354, 696, 742, 1084], # 1
        [776, 684,   0, 992, 878, 502, 274, 810,  468, 742, 400], # 2
        [696, 308, 992,   0, 114, 650, 878, 502, 844, 890, 1232], # 3
        [582, 194, 878, 114,   0, 536, 764, 388, 730, 776, 1118], # 4
        [274, 502, 502, 650, 536,   0, 228, 308, 194, 240, 582],  # 5
        [502, 730, 274, 878, 764, 228,   0, 536, 194, 468, 354],  # 6
        [194, 354, 810, 502, 388, 308, 536,   0, 342, 388, 730],  # 7
        [308, 696, 468, 844, 730, 194, 194, 342,   0, 274, 388],  # 8
        [194, 742, 742, 890, 776, 240, 468, 388, 274,   0, 342],  # 9
        [536,1084, 400,1232,1118, 582, 354, 730, 388, 342,   0],  # 10
    ]
    data["num_vehicles"] = 4
    data["depot"] = 0
    data["demands"] = [0, 1, 1, 3, 6, 3, 6, 5, 3, 1, 2]
    data["vehicle_capacities"] =  [10]*4
    return data

In [3]:
data = create_data_model()

#Create the routing index manager
manager = pywrapcp.RoutingIndexManager(
    len(data["distance_matrix"]), data["num_vehicles"], data["depot"]) 

#Create the routing index manager  
routing = pywrapcp.RoutingModel(manager) 

In [4]:
#Create and register a distance callback
def distance_callback(from_index, to_index):
    """Returns the distance between the two nodes."""
    # Convert from routing variable Index to distance matrix NodeIndex.
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return data["distance_matrix"][from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)

In [5]:
#Define cost of each arc
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

In [6]:
#Create and register a demand callback
def demand_callback(from_index):
    """Returns the demand of the node."""
    # Convert from routing variable Index to demands NodeIndex.
    from_node = manager.IndexToNode(from_index)
    return data["demands"][from_node]

demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)

In [7]:
#The Vehicle Capacity Dimension
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index,
    0,  # null capacity slack
    data["vehicle_capacities"],  # vehicle maximum capacities
    True,  # start cumul to zero
    "Capacity",
)

True

In [8]:
def print_solution(data, manager, routing, solution):
    """Prints the route for each vehicle and summary stats."""
    print("\n" + "="*55)
    print("  CVRP SOLUTION")
    print("="*55)

    total_distance = 0
    total_load = 0

    for vehicle_id in range(data["num_vehicles"]):
        index = routing.Start(vehicle_id)
        route_distance = 0
        route_load = 0
        route_output = f"\n  Vehicle {vehicle_id + 1} route:\n  "

        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_load += data["demands"][node_index]
            route_output += f" Site {node_index} (load:{route_load}) ->"
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(
                previous_index, index, vehicle_id
            )

        # Add the depot at the end
        route_output += f" Depot (Site 0)"
        print(route_output)
        print(f"     Distance: {route_distance} km")
        print(f"     Load:     {route_load} / {data['vehicle_capacities'][vehicle_id]} units")
        total_distance += route_distance
        total_load += route_load

    print("\n" + "-"*55)
    print(f"  Total distance across all vehicles : {total_distance} km")
    print(f"  Total demand served                : {total_load} units")
    print("="*55 + "\n")


In [9]:
#Phase 1
#Setting first solution heuristic (Nearest neighbour method)
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)

#Phase 2 - improve the first solution
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH)
search_parameters.time_limit.seconds = 5
#search_parameters.log_search = True


In [ ]:
#Solve the problem
solution = routing.SolveWithParameters(search_parameters)

In [ ]:
#Print the solution to console 
if solution:
    print_solution(data,manager, routing, solution)
else:
        print("No solution found !")